# Relatório de lojas e manutenções no mês de Dez/2025

## Área de Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import folium
from folium.plugins import Search
import geopandas as gpd

## Arquivo de Relatório de Requisições do Mercado Eletrônico - Dezembro/2025

In [ ]:
mercado_eletronico = pd.read_excel('Relatório_ME_DEZ_2025.xlsx')
mercado_eletronico.head()

In [ ]:
mercado_eletronico.info()

In [ ]:
mercado_eletronico.count()

In [ ]:
mercado_eletronico.columns

## Remoção de colunas desnecessárias

In [ ]:
colunas_para_remover = [
    'Conta Contábil', 'CiaERP', 'SV', 'Depto. Solicitante', 'Requisição Cliente', 'Ordem Estatística', 'Descrição Ordem Estatística',
    'Ordem Investimento', 'Descrição Ordem Investimento', 'Data Prevista', 'Solicitante','WFC - BANCO AGIBANK S.A'
]
mercado_eletronico.drop(columns=colunas_para_remover, inplace=True)

In [ ]:
mercado_eletronico.count()

In [ ]:
mercado_eletronico.drop(columns='Local de Cobrança', inplace=True)

## Verificação de dados

In [ ]:
mercado_eletronico.count()

In [ ]:
mercado_eletronico.head()

In [ ]:
mercado_eletronico['PROJETO'].count()

In [ ]:
mercado_eletronico['Categoria da Requisição'].value_counts().sort_values()

In [ ]:
mercado_eletronico.columns

In [ ]:
mercado_eletronico['Preço Total Pedido c/ Impostos'].count()

## Chamados da manutenção

In [ ]:
manutencao_geral = mercado_eletronico[
    ((mercado_eletronico['Categoria da Requisição'] == 'Manutenção') |
    (mercado_eletronico['Categoria da Requisição'] == 'EMERGENCIAL')) &
    (mercado_eletronico['PROJETO'] == 'NA-PROJETO')
]

manutencao_geral.count()

In [ ]:
manutencao_geral.head()

In [ ]:
mercado_eletronico.loc[
    (mercado_eletronico['Categoria da Requisição'] == 'Manutenção') |
    (mercado_eletronico['Categoria da Requisição'] == 'EMERGENCIAL'),
    'Pedido BANN'
]

## Informações gerais sobre as lojas do agi

In [ ]:
locais = pd.read_excel('Endereço das lojas_2025.xlsx')
locais.head()

In [ ]:
locais.drop(columns='COM REQUISIÇÃO ABERTA PAINEL DE SENHA', inplace=True)

In [ ]:
locais.columns

In [ ]:
mercado_eletronico.rename(columns={'C. Custo': 'CC'}, inplace=True)

In [ ]:
mercado_eletronico.drop(columns={'Local de Entrega',
                                'Local de Faturamento',
                                }, inplace=True)

In [ ]:
mercado_eletronico.columns

## Junção de tabelas

In [ ]:
mercado_eletronico['CC'] = locais['CC'].astype(str)
locais['CC'] = locais['CC'].astype(str)

mercado_eletronico_ajustado = pd.merge(mercado_eletronico, locais, on='CC', how='left')
mercado_eletronico_ajustado = (mercado_eletronico_ajustado[((mercado_eletronico_ajustado['Categoria da Requisição'] == 'Manutenção') |
                              (mercado_eletronico_ajustado['Categoria da Requisição'] == 'EMERGENCIAL')) &
                              (mercado_eletronico_ajustado['PROJETO'] == 'NA-PROJETO')])

mercado_eletronico_ajustado.count()

In [ ]:
mercado_eletronico_ajustado.head()

## Contagem de chamados por categoria

In [ ]:
contagem_categoria = mercado_eletronico_ajustado['Tipo de Requisição'].value_counts().sort_values(ascending=True)
contagem_categoria.head()

In [ ]:
ax1 = contagem_categoria.plot(
    kind='bar',
    figsize=(20,7),
    color={"#27A9FF","#FF5733"},
    title='Quantidade de categoria por requisição'
)
ax1.set_ylabel('')

## Contagem de requisições por solicitante

In [ ]:
contagem_nome_solicitante_manut_normal = (
    mercado_eletronico_ajustado
    .loc[mercado_eletronico_ajustado['Categoria da Requisição'] == 'Manutenção', 'Nome do Solicitante']
    .value_counts()
    .sort_values(ascending=True)
)

contagem_nome_solicitante_manut_normal.sort_values(ascending=False).head()

In [ ]:
# identifica o item com maior valor
max_index = contagem_nome_solicitante_manut_normal.idxmax()

# Cria lista de cores: uma cor para todos, outra para o máximo
cores = ['#27A9FF' if i != max_index else '#FF5733'  # cor diferente p/ o maior
         for i in contagem_nome_solicitante_manut_normal.index]


ax1 = contagem_nome_solicitante_manut_normal.plot(
    kind='barh',
    figsize=(20,10),
    color=cores,
    title='Quantidade por nome solicitante',
    ylabel='',
)
ax1.set_ylabel('')

## Lojas Agi

In [ ]:
locais_reduzido = locais[['CC', 'Latitude', 'Longitude']]
locais_reduzido

In [ ]:
gdf = gpd.GeoDataFrame(
    locais_reduzido, 
    geometry=gpd.points_from_xy(locais_reduzido['Longitude'], locais_reduzido['Latitude']),
    crs="EPSG:4326"
)

## Mapa de lojas agi Brasil

In [ ]:
mapa = folium.Map(
    location=[locais['Latitude'].mean(), locais['Longitude'].mean()],
    zoom_start=5,
    tiles="CartoDB positron"
)

search_layer = folium.GeoJson(
    gdf,
    name="Busca Invisivel",
    marker=folium.CircleMarker(radius=0, fill_opacity=0, opacity=0), 
    tooltip=None,
    style_function=lambda x: {'opacity': 0, 'fillOpacity': 0}
).add_to(mapa)

visual_layer = folium.FeatureGroup(name="Lojas")

for idx, row in locais.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=3,
        color='blue',
        fill=True,
        fill_color='red',
        fill_opacity=0.8,
        tooltip=str(row['CC']),
        ).add_to(visual_layer)

visual_layer.add_to(mapa)

Search(
    layer=search_layer,
    geom_type='Point',
    placeholder='Pesquisar CC...',
    collapsed=False,
    search_label='CC',
    weight=3,
    zoom=18
).add_to(mapa)

mapa.save("mapa_lojas.html")
mapa

In [ ]:
fig = px.density_mapbox(locais, 
                        lat='Latitude', 
                        lon='Longitude', 
                        radius=10,       # Raio de influência do calor
                        zoom=2.3,
                        hover_name="CC",
                        hover_data=["CIDADE", "REGIONAL", "COORDENAÇÃO"],
                        mapbox_style="open-street-map",
                        title="Mapa de Calor: Concentração de Lojas")

fig.show()

## Quantidade de lojas x requisição

Aqui conseguimos ver as lojas que foram abertas requisições de manutenção no mês de dezembro.

Como podemos ver, a maior parte foi na região suldeste e a menor parte na região sul

* Obs: Temos uma baixa concetração de lojas na região Nortem como vimos no mapa de calor

In [ ]:
fig_lojas_agi_x_requisicao = px.scatter_map(mercado_eletronico_ajustado,
    lat="Latitude",
    lon="Longitude",    # Segmentação por cor
    hover_name="CC",
    hover_data=["CIDADE", "REGIONAL", "COORDENAÇÃO"],
    color="REGIONAL",
    zoom=2.5,
    height=600,
    size_max=30 )

#Configurando o estilo do mapa (OpenStreetMap é gratuito e não precisa de API Key)
fig_lojas_agi_x_requisicao.update_traces(marker=dict(size=10))
fig_lojas_agi_x_requisicao.update_layout(mapbox_style="open-street-map")
fig_lojas_agi_x_requisicao.update_layout(title="Lojas Agi x Requisição")

fig_lojas_agi_x_requisicao.show()

## Análise de fornecedores que nos atenderam em DEZ/2025

Como podemos ver, tivemos em torno de 17 fornecedores nos atendendo para manutenções no mês de Dezembro, com um destaque maior para o ESTEVES GUIMARAES CONSTRUCOES LTDA, com 158 requisições e um valor total de R$ 294,829.92.

In [ ]:
analise_preco_fornecedor = mercado_eletronico_ajustado.groupby('Fornecedor')['Preço Total Pedido c/ Impostos'].agg(
    Contagem=np.size,               # Quantos pedidos tem?
    Preco_Total=np.sum,             # Valor total adquirido
    Preco_Minimo=np.min,            # Mais barato
    Preco_Medio=np.mean,            # Média (Cuidado!)
    Preco_Mediano=np.median,        # Mediana (A realidade do mercado)
    Preco_Maximo=np.max,            # Mais caro (Outlier)
    Desvio_Padrao=np.std            # Variação
).sort_values('Preco_Total', ascending=False)

# 2. FORMATANDO COMO PLANILHA (Styler)
# Isso gera uma visualização HTML idêntica ao Excel com formatação condicional
formatacao = {
    'Preco_Total': 'R$ {:,.2f}',
    'Preco_Minimo': 'R$ {:,.2f}',
    'Preco_Medio': 'R$ {:,.2f}',
    'Preco_Mediano': 'R$ {:,.2f}',
    'Preco_Maximo': 'R$ {:,.2f}',
    'Desvio_Padrao': 'R$ {:,.2f}'
}

print("=== RELATÓRIO DE PRECIFICAÇÃO POR FORNECEDOR ===")
display(
    analise_preco_fornecedor.style
    .format(formatacao)
    # Gradiente de cor para destacar os valores altos (Heatmap)
    .background_gradient(subset=['Preco_Mediano', 'Preco_Maximo'], cmap='Reds')
    .background_gradient(subset=['Contagem'], cmap='Blues')
    .background_gradient(subset=['Preco_Total'], cmap='Oranges')

)

## Exportação de DataFrame limpo e tratado para um arquivo CSV

In [ ]:
mercado_eletronico_ajustado.to_csv('mercado_eletrônico_DEZ_2025.csv', index=False)